# Exploratory Data Analysis & Model Training Walkthrough

This notebook documents the end-to-end training and evaluation process of our **Document Categorization and Tagging** project.

We cover:
1. **Data Ingestion**: Loading multilingual reviews from `buruzaemon/amazon_reviews_multi`.
2. **Text Preprocessing**: Cleaning, emoji stripping, and tokenizing raw reviews.
3. **Baseline Modeling**: TF-IDF + Logistic Regression binary sentiment classifier.
4. **Transfer Learning**: Fine-tuning TFDistilBertForSequenceClassification in Keras.

## 1. Ingestion and Exploratory Data Analysis (EDA)

We load production review records utilizing our balanced dataset loaders.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from utils.data_loader import load_production_dataset
import pandas as pd

# Ingest 1000 reviews for rapid EDA
df = load_production_dataset(sample_size=1000)
print(f"Dataset columns: {df.columns.tolist()}")
print(f"Language distribution:\n{df['language'].value_counts()}")
print(f"Category rating distribution:\n{df['category'].value_counts()}")
df.head()

## 2. Text Preprocessing and Normalization

Applying multi-language cleaning filters (punctuation normalization, HTML tag removals, and SpaCy pipeline pipelines).

In [ ]:
from utils.text_preprocessing import TextPreprocessor

preprocessor = TextPreprocessor(default_lang='en')
df['cleaned_text'] = df['text'].apply(preprocessor.clean_raw_text)
df[['text', 'cleaned_text']].head()

## 3. Baseline ML Model Training

Train a traditional TF-IDF + Logistic Regression classifier on binary sentiment targets (1-2 vs. 4-5 stars).

In [ ]:
from models.baseline_classifier import train_and_evaluate_baseline
train_and_evaluate_baseline()

## 4. Deep Learning DistilBERT Fine-Tuning

Fine-tune a Hugging Face pre-trained DistilBERT sequence classifier utilizing custom training loop with Adam optimizer and validation callbacks.

In [ ]:
from models.text_classifier import run_production_training
run_production_training(sample_size=200, epochs=3, batch_size=32, run_eagerly=True)